In [0]:
# =============================================================================
# Notebook  : _map_common
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/_map_common
# Purpose   : Shared utilities for all 02b Map layer notebooks.
#
# Every map notebook does:
#   Cell 1:  %run ../config/pipeline_config    ← loads all config variables
#   Cell 2:  %run ./_map_common               ← loads map utilities
#
# REQUIRES pipeline_config to be %run FIRST — uses these variables:
#   sv, DELTA_TBLPROPS_MAP, SPARK_CONF_MAP, DQ_THRESHOLDS, FREE_EMAIL_DOMAINS
# =============================================================================

from pyspark.sql import functions as F

# =============================================================================
# 0. Serverless-safe Spark config
#    On Serverless: spark.databricks.* and spark.sql.shuffle.partitions
#    are silently skipped — Serverless manages these automatically.
#    On Job Cluster: all configs apply for maximum performance.
# =============================================================================
def safe_spark_conf(configs: dict):
    """Set spark configs; silently ignore any not supported on this compute."""
    for k, v in configs.items():
        try:
            spark.conf.set(k, v)
        except Exception:
            pass   # Serverless rejects some — completely safe to ignore

safe_spark_conf(SPARK_CONF_MAP)   # from pipeline_config

# =============================================================================
# 1. Map Table Creation Helper
# =============================================================================
def create_map_table(table_full: str, schema_def: str, cluster_by: str = None):
    """Creates a Delta map output table if it does not already exist."""
    cluster_clause = f"CLUSTER BY ({cluster_by})" if cluster_by else ""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_full}
        ({schema_def})
        USING DELTA
        {cluster_clause}
        {DELTA_TBLPROPS_MAP}
    """)
    print(f"  Table ready: {table_full}")

# =============================================================================
# 2. Count Helper
# =============================================================================
def cnt(table_full: str) -> int:
    return spark.table(table_full).count()

# =============================================================================
# 3. Safe DROP VIEW helper
# =============================================================================
def safe_drop_view(table_full: str):
    """Drop a VIEW if it exists; silently ignore if it's a TABLE or doesn't exist.
    Prevents DELTA_CANNOT_WRITE_INTO_VIEW on CREATE OR REPLACE TABLE."""
    try:
        spark.sql(f"DROP VIEW IF EXISTS {table_full}")
    except Exception:
        pass

print("_map_common loaded.")